# NPZ Gaze Visualization Viewer

This notebook visualizes SparseGaze per-sequence prediction `.npz` files with the same output types used by CSV gaze visualization:

- Scene-frame gaze rays
- reference-frame scanpath overlay
- clean reference-frame scanpath
- per-frame RGB overlays
- overlay video

Important: prediction `.npz` files store `pred_xyz` / `gt_xyz` as **Scene-frame unit gaze directions**. They do not store depth-defined gaze points. This notebook uses extracted `gaze_samples.csv` to recover the gaze/head origin and either:

- `gt_depth`: place the predicted ray endpoint at the GT ADT gaze depth for that frame;
- `fixed`: place every endpoint at a fixed metric ray length.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

from IPython.display import Image, Video, display

from adt_sandbox.config import load_dotenv
from adt_sandbox.providers import create_adt_providers
from visualization.npz_gaze_outputs import (
    generate_npz_gaze_visualizations,
    load_prediction_npz,
    prediction_metadata_table,
)
from visualization.prediction_eval import discover_prediction_runs

load_dotenv(REPO_ROOT / ".env")

## Configuration

Set `EVAL_ROOT` to a model eval directory that contains `sequence_predictions/*.npz`. The `sparsegaze_cpf_rotation_only` aggregate directory may not contain per-sequence NPZ files unless that eval was run with sequence prediction saving enabled.

In [ ]:
REPORTS_DIR = Path("/mnt/d/SparseGaze/ADT-Gaze-structured")
EVAL_ROOT = Path("/home/liumu/Github_Projects/SparseGaze/outputs/eval/adt/sparsegaze")
OUTPUT_ROOT = REPO_ROOT / "outputs" / "figures" / "gaze_npz"

runs = discover_prediction_runs(EVAL_ROOT)
print(f"eval_root={EVAL_ROOT}")
print(f"prediction files={len(runs)}")
display(runs.head(80))

In [ ]:
# Selection. Edit these after inspecting `runs`.
SELECT_EVAL_KIND = runs["eval_kind"].iloc[0] if not runs.empty else "rollout"
SELECT_SEQUENCE = runs["sequence"].iloc[0] if not runs.empty else ""
SELECT_HZ = int(runs["target_hz"].iloc[0]) if not runs.empty else 10
SELECT_PHASE = int(runs["phase"].iloc[0]) if not runs.empty else 0

# Track can be "pred" or "gt". Run both tracks in separate cells if needed.
TRACK = "pred"
DEPTH_MODE = "gt_depth"  # "gt_depth" or "fixed"
FIXED_DEPTH_M = 2.0
START_ROW = 0
END_ROW = 120
STRIDE = 10

selected = runs[
    (runs["eval_kind"] == SELECT_EVAL_KIND)
    & (runs["sequence"] == SELECT_SEQUENCE)
    & (runs["target_hz"] == SELECT_HZ)
    & (runs["phase"] == SELECT_PHASE)
]
if selected.empty:
    raise ValueError("No prediction NPZ matches the current selection.")
NPZ_PATH = Path(selected.iloc[0]["path"])
print(NPZ_PATH)
display(prediction_metadata_table(NPZ_PATH))

## Generate CSV-Style Visualizations from NPZ

This opens the ADT provider only for the selected sequence/window. `skeleton_flag=False` is intentional: these gaze visualizations only need RGB frames, poses, and calibration.

In [ ]:
prediction = load_prediction_npz(NPZ_PATH)
providers = create_adt_providers(prediction["sequence"], skeleton_flag=False)

result = generate_npz_gaze_visualizations(
    gt_provider=providers.gt_provider,
    npz_path=NPZ_PATH,
    reports_dir=REPORTS_DIR,
    output_root=OUTPUT_ROOT,
    track=TRACK,
    start_row=START_ROW,
    end_row=END_ROW,
    stride=STRIDE,
    depth_mode=DEPTH_MODE,
    fixed_depth_m=FIXED_DEPTH_M,
    run_name=f"{SELECT_EVAL_KIND}_hz{SELECT_HZ}_phase{SELECT_PHASE}_{TRACK}_{DEPTH_MODE}_{START_ROW}_{END_ROW}_s{STRIDE}",
)
result

## Display Generated Figures

The filenames mirror the CSV visualizer outputs. `pred_*` and `gt_*` outputs can be generated separately by changing `TRACK` and rerunning the previous cell.

In [ ]:
out_dir = Path(result["output_dir"])
print(out_dir)

for name in [
    f"{TRACK}_scene_rays.png",
    f"{TRACK}_reference_frame_scanpath_overlay.png",
    f"{TRACK}_reference_frame_scanpath_clean.png",
]:
    path = out_dir / name
    print(path)
    if path.exists():
        display(Image(filename=str(path)))

In [ ]:
video_path = out_dir / f"{TRACK}_overlay_video.mp4"
print(video_path)
if video_path.exists():
    display(Video(str(video_path), embed=True))

## Compare Pred and GT Tracks

To compare predicted and GT visualization outputs, rerun the generation cell twice:

1. `TRACK = "gt"`
2. `TRACK = "pred"`

Both are saved under the same `OUTPUT_ROOT`, with the track name included in filenames and run names.

The current adapter uses Scene-frame directions from the NPZ. It does not convert them back to CPF yaw/pitch. For CPF-space residual analysis, use `notebooks/08_prediction_gaze_evaluation_viewer.ipynb`.